# Experiment 03: Runtime Dependency Analysis

## Objective

The objective of this experiment is to investigate the runtime dependency chain of the Snapdragon-enabled `llama.cpp` binaries on Android and identify the root cause preventing successful execution.

Previous experiments confirmed that:

- The Snapdragon backend compiles successfully.
- The deployment package is generated correctly.
- All binaries and shared libraries are transferred successfully to the Android device.

However, execution initially failed during Android's dynamic linking stage with a runtime linker error.

This experiment performs a systematic investigation of executable dependencies, shared library requirements, runtime environment configuration, and Android linker behavior to identify the source of the failure.

## Initial Runtime Failure

The deployed binary was executed on the Android device using the Snapdragon runtime environment.

Environment variables:

```bash
export LD_LIBRARY_PATH=/data/local/tmp/llama/lib:/vendor/lib64:/system/lib64
export ADSP_LIBRARY_PATH=/vendor/lib/rfsa/adsp:/system/lib/rfsa/adsp:/data/local/tmp/llama/lib
```

Execution:
```bash
./bin/llama-cli --help
```

Observed error:
```text
CANNOT LINK EXECUTABLE "./bin/llama-cli":
cannot locate symbol
_ZNSt3__113__hash_memoryEPKvm
referenced by
/system/lib64/liblog.so
```

Since the executable failed before entering main(), the investigation focused on Android's dynamic linker and runtime dependency resolution.

## Investigation Strategy

The investigation followed a bottom-up approach:

1. Inspect executable dependencies.
2. Inspect backend shared library dependencies.
3. Verify Android program interpreter.
4. Search for unresolved symbols in shipped libraries.
5. Inspect Android system libraries.
6. Verify deployment package integrity.
7. Isolate runtime environment variables.
8. Validate corrected runtime configuration.

Each hypothesis was either confirmed or rejected using direct evidence.

## Investigation 1 — Executable Dependencies

The first step was to inspect the shared library dependencies of the generated executable using `readelf`.

Purpose:

- Verify that all required shared libraries are present.
- Detect missing runtime dependencies before execution.

```bash
readelf -d build-snapdragon/bin/llama-cli | grep NEEDED
```

### Observation

The executable depends only on the expected `llama.cpp` runtime libraries together with standard Android system libraries.

No unexpected dependencies were discovered.

This confirms that the executable itself was built correctly.

## Investigation 2 — Snapdragon Backend Shared Library Dependencies

The executable itself showed no abnormal runtime dependencies.

The next step was to inspect the Snapdragon backend implementation (`libggml-hexagon.so`) to determine whether the backend introduced additional shared library requirements that could explain the linker failure.

The library was inspected using `readelf` to analyse its dynamic dependency table.

```bash
readelf -d pkg-snapdragon/llama.cpp/lib/libggml-hexagon.so
```

### Observation

The Snapdragon backend depends on the following shared libraries:

- `liblog.so`
- `libdl.so`
- `libggml-base.so`
- `libm.so`
- `libc.so`

No dependency on `libc++.so` was observed.

Additionally, the library does not define any runtime search paths (`RPATH` or `RUNPATH`).

### Conclusion

The Snapdragon backend does not explicitly depend on the Android C++ runtime library.

This indicates that the unresolved symbol reported by the Android linker is unlikely to originate from the backend's declared ELF dependencies.

The investigation therefore proceeds to inspect how the executable is loaded by the Android runtime.

## Investigation 3 — Android Program Interpreter

The executable and backend dependencies appeared to be valid.

The next step was to determine which runtime loader was responsible for loading the executable.

On Android, ELF executables specify a program interpreter (dynamic linker) responsible for resolving shared library dependencies during process initialization.

The interpreter was inspected using `readelf`.

```bash
readelf -l build-snapdragon/bin/llama-cli | grep interpreter
```

### Observation

The executable specifies the following program interpreter:

```
/system/bin/linker64
```

This is Android's native 64-bit dynamic linker responsible for loading ELF executables and resolving runtime dependencies.

### Conclusion

The executable relies on Android's standard dynamic linker rather than a custom runtime loader.

This confirms that the linker error originates during Android's normal shared library resolution process.

The investigation therefore shifted towards identifying which runtime library introduced the unresolved symbol reported by the linker.

## Investigation 4 — Search for the Missing Symbol in the Deployment Package

The linker reported that the symbol

`_ZNSt3__113__hash_memoryEPKvm`

could not be resolved.

To determine whether the deployment package itself introduced this dependency, every shared library was inspected for references to the reported symbol.

The search was performed using `readelf`.

```bash
for f in pkg-snapdragon/llama.cpp/lib/*.so; do
    echo "=================================================="
    echo "$f"
    echo "=================================================="
    readelf -Ws "$f" | grep "_ZNSt3__113__hash_memoryEPKvm"
done
```

### Observation

The symbol `_ZNSt3__113__hash_memoryEPKvm` was not referenced by any shared library included in the deployment package.

All deployed libraries completed the inspection without producing a matching symbol reference.

### Conclusion

The unresolved symbol does not originate from the Snapdragon deployment package.

This eliminates the application binaries as the direct source of the linker failure and suggests that the dependency originates from Android's system runtime.

The next investigation therefore focused on the Android system libraries involved in the reported linker error.

## Investigation 5 — Analysis of Android System Library (`liblog.so`)

The Android linker reported that the unresolved symbol originated from the system library `liblog.so`.

To determine whether the issue was caused by the application or the Android runtime, the system library was extracted from the target device and inspected using `readelf`.

The objective was to determine:

- whether `liblog.so` references the missing symbol, and
- which runtime libraries it depends upon.

```bash
adb pull /system/lib64/liblog.so .
```

```bash
readelf -Ws liblog.so | grep "_ZNSt3__113__hash_memoryEPKvm"
```

```bash
readelf -d liblog.so
```

### Observation

Inspection of `liblog.so` revealed that:

- The symbol `_ZNSt3__113__hash_memoryEPKvm` is referenced as an **undefined** symbol.
- The library declares `libc++.so` as a required shared library.

This indicates that `liblog.so` expects the symbol to be resolved by the Android C++ runtime during dynamic linking.

### Conclusion

The linker error does not originate from `liblog.so` itself.

Instead, `liblog.so` relies on `libc++.so` to provide the missing symbol.

This shifted the investigation towards verifying whether the required symbol was actually available in the Android C++ runtime.

## Investigation 6 — Verification of Android C++ Runtime (`libc++.so`)

Since `liblog.so` depends on `libc++.so`, the next step was to verify whether the reported symbol was actually present in the Android C++ runtime.

If the symbol existed, the failure would likely be caused by runtime library resolution rather than a missing library.

```bash
adb pull /system/lib64/libc++.so .
```

```bash
readelf -Ws libc++.so | grep "_ZNSt3__113__hash_memoryEPKvm"
```

```bash
readelf -d libc++.so
```

### Observation

Inspection of `libc++.so` confirmed that the symbol

`_ZNSt3__113__hash_memoryEPKvm`

is exported by the Android C++ runtime.

The library also exposes the expected SONAME (`libc++.so`) together with its standard Android runtime dependencies.

### Conclusion

The required symbol is present on the Android device.

This eliminates the possibility that the linker failure is caused by a missing C++ runtime library.

The investigation therefore shifted towards understanding why Android's dynamic linker failed to resolve a symbol that is already available in the system runtime.

### Conclusion

The required symbol is present on the Android device.

This eliminates the possibility that the linker failure is caused by a missing C++ runtime library.

The investigation therefore shifted towards understanding why Android's dynamic linker failed to resolve a symbol that is already available in the system runtime.

### Test Configuration A — Snapdragon Runtime Environment

The initial runtime environment followed the Snapdragon backend setup procedure.

```bash
export LD_LIBRARY_PATH=/data/local/tmp/llama/lib:/vendor/lib64:/system/lib64
export ADSP_LIBRARY_PATH=/vendor/lib/rfsa/adsp:/system/lib/rfsa/adsp:/data/local/tmp/llama/lib
```

To verify the runtime environment, a standard Android executable (`ls`) was executed before running the application.

```bash
ls
```

### Observation

Executing `ls` produced the following linker error:

```
CANNOT LINK EXECUTABLE "ls":
cannot locate symbol "_ZNSt3__113__hash_memoryEPKvm"
referenced by "/system/lib64/liblog.so"
```

This demonstrated that the issue was **not limited to `llama.cpp`**.

Even standard Android executables failed to start after configuring the runtime environment.

### Test Configuration B — Isolated Library Path

To isolate the cause, the runtime environment was simplified by exposing only the deployment library directory.

```bash
export LD_LIBRARY_PATH=/data/local/tmp/llama/lib

export ADSP_LIBRARY_PATH=/vendor/lib/rfsa/adsp:/system/lib/rfsa/adsp:/data/local/tmp/llama/lib
```

```bash
ls
```

### Observation

With the simplified `LD_LIBRARY_PATH`, the `ls` command executed successfully.

This demonstrated that the deployment package itself was not responsible for the linker failure.

Instead, the issue appeared only when `/vendor/lib64` and `/system/lib64` were explicitly appended to `LD_LIBRARY_PATH`.

```bash
./bin/llama-cli --help
```

### Observation

Under the simplified runtime configuration, `llama-cli` initialized successfully.

The executable progressed beyond Android's dynamic linking stage, initialized the Hexagon backend, and displayed the command-line help.

The previous linker error was no longer observed.

The remaining runtime message was:

```
ggml-hex: failed to open session 0 : error 0x80000406
ggml-hex: failed to create device/session 0
```

Unlike the earlier failure, this message originated from the Qualcomm Hexagon runtime after successful executable initialization.

### Conclusion

The runtime dependency investigation identified the root cause of the original linker failure.

The deployment package, executable, and Android runtime libraries were all valid.

The failure was caused by the runtime environment configuration, specifically the explicit inclusion of `/vendor/lib64` and `/system/lib64` in `LD_LIBRARY_PATH`.

Restricting `LD_LIBRARY_PATH` to the deployment library directory allowed Android's native linker to correctly resolve system libraries, eliminating the unresolved symbol error.

The investigation therefore concludes that the original runtime failure was caused by an incorrect library search path configuration rather than missing binaries or unresolved dependencies.

# Experiment Summary

## Key Findings

The following conclusions were established during this experiment:

- The deployment package was complete and correctly transferred to the Android device.
- The executable and Snapdragon backend declared valid runtime dependencies.
- The reported C++ runtime symbol existed in Android's `libc++.so`.
- The linker failure did not originate from the deployment package.
- Explicitly extending `LD_LIBRARY_PATH` with Android system library directories altered runtime linker behaviour.
- Restricting `LD_LIBRARY_PATH` to the deployment library directory resolved the linker failure.

Following this correction, the application successfully initialized and reached Qualcomm Hexagon runtime initialization.

The remaining `0x80000406` session creation error is unrelated to Android's dynamic linker and will be investigated in the next experiment.

# Lessons Learned

This experiment demonstrates the importance of validating the runtime environment independently of the application.

Although the initial linker error appeared to indicate a missing C++ runtime symbol, systematic investigation showed that:

- the deployment package was complete,
- the Android runtime contained the required symbol,
- and the failure was caused by the runtime library search path configuration.

Restricting `LD_LIBRARY_PATH` to the deployment library directory restored correct linker behaviour and allowed the application to initialize successfully.

This investigation also illustrates the value of isolating environmental variables during runtime debugging before modifying application binaries or rebuilding the software.